[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_stable_distributions/SFM_ch2_stable_distributions.ipynb)

# SFM_ch2_stable_distributions

Stable Distributions for Financial Returns
Description:
- Stable distribution PDFs for various alpha values
- S&P 500 stable fit vs Normal (log-scale)
- VaR comparison: stable vs normal vs empirical
- Survival function (log-log) tail comparison
- Cross-asset stability index comparison
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from scipy.stats import levy_stable
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Color palette
MAIN_BLUE = '#1A3A6E'
CRIMSON = '#DC3545'
FOREST = '#2E7D32'
AMBER = '#B5853F'
ORANGE = '#E67E22'

# Output directory
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Stable Distribution PDFs

In [ ]:
# Stable PDFs for various alpha values
x = np.linspace(-6, 6, 1000)

# Parameters: (alpha, beta=0, loc=0, scale=1)
alphas = [0.5, 1.0, 1.5, 2.0]
labels = [r'$\alpha=0.5$ (L\'{e}vy)', r'$\alpha=1.0$ (Cauchy)',
          r'$\alpha=1.5$', r'$\alpha=2.0$ (Gaussian)']
colors = [CRIMSON, ORANGE, FOREST, MAIN_BLUE]
styles = ['-', '--', '-.', '-']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Panel A: Linear scale
for alpha, label, color, ls in zip(alphas, labels, colors, styles):
    pdf = levy_stable.pdf(x, alpha, 0, loc=0, scale=1)
    ax1.plot(x, pdf, color=color, linewidth=1.2, linestyle=ls, label=label)

ax1.set_xlabel('$x$')
ax1.set_ylabel('$f(x)$')
ax1.set_title('Ch.2: Stable PDFs (linear scale)', fontweight='bold', fontsize=9)
ax1.set_xlim(-6, 6)
ax1.set_ylim(0, 0.55)
ax1.legend(loc='upper right', fontsize=7, frameon=False)

# Panel B: Log scale
for alpha, label, color, ls in zip(alphas, labels, colors, styles):
    pdf = levy_stable.pdf(x, alpha, 0, loc=0, scale=1)
    pdf_clipped = np.clip(pdf, 1e-10, None)
    ax2.plot(x, pdf_clipped, color=color, linewidth=1.2, linestyle=ls, label=label)

ax2.set_yscale('log')
ax2.set_xlabel('$x$')
ax2.set_ylabel('$f(x)$ (log scale)')
ax2.set_title('Ch.2: Stable PDFs (log scale)', fontweight='bold', fontsize=9)
ax2.set_xlim(-6, 6)
ax2.set_ylim(1e-5, 1)
ax2.legend(loc='lower center', fontsize=7, frameon=False)

plt.tight_layout()
save_fig('ch2_stable_pdfs')

## S&P 500 Stable Fit

In [ ]:
# Download S&P 500 data
data = yf.download('^GSPC', start='2000-01-01', end='2025-01-01', progress=False)
close = data['Close'].squeeze()
log_ret = np.log(close / close.shift(1)).dropna()

print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(log_ret)}")

# Fit stable distribution
# levy_stable.fit can be slow; use _fitstart for initial params then refine
pars_init = levy_stable._fitstart(log_ret)
stable_params = levy_stable.fit(log_ret, floc=pars_init[2], fscale=pars_init[3])
alpha_hat, beta_hat, loc_hat, scale_hat = stable_params

print(f"   Stable fit:")
print(f"     alpha = {alpha_hat:.4f}")
print(f"     beta  = {beta_hat:.4f}")
print(f"     loc   = {loc_hat:.6f}")
print(f"     scale = {scale_hat:.6f}")

# Fit Normal
mu_hat, sigma_hat = stats.norm.fit(log_ret)
print(f"   Normal fit:")
print(f"     mu    = {mu_hat:.6f}")
print(f"     sigma = {sigma_hat:.6f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))

# Histogram
counts, bins, _ = ax.hist(log_ret, bins=200, density=True, alpha=0.5,
                           color=MAIN_BLUE, edgecolor='white', linewidth=0.3,
                           label='S&P 500 log-returns')

# Overlay PDFs
x_plot = np.linspace(log_ret.min(), log_ret.max(), 1000)
pdf_stable = levy_stable.pdf(x_plot, alpha_hat, beta_hat,
                              loc=loc_hat, scale=scale_hat)
pdf_normal = stats.norm.pdf(x_plot, mu_hat, sigma_hat)

ax.plot(x_plot, pdf_stable, color=CRIMSON, linewidth=1.5,
        label=f'Stable ($\\alpha$={alpha_hat:.2f}, $\\beta$={beta_hat:.2f})')
ax.plot(x_plot, pdf_normal, color=FOREST, linewidth=1.2, linestyle='--',
        label=f'Normal ($\\mu$={mu_hat:.4f}, $\\sigma$={sigma_hat:.4f})')

ax.set_yscale('log')
ax.set_ylim(1e-2, None)
ax.set_xlabel('Log-return')
ax.set_ylabel('Density (log scale)')
ax.set_title('Ch.2: S&P 500 Stable Fit vs Normal', fontweight='bold', fontsize=10)

# Annotation box
info_text = (f'$\\alpha$ = {alpha_hat:.3f}\n'
             f'$\\beta$ = {beta_hat:.3f}\n'
             f'n = {len(log_ret):,}')
ax.text(0.97, 0.97, info_text, transform=ax.transAxes, ha='right', va='top',
        fontsize=8, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3,
          frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_sp500_stable_fit')

## VaR Comparison

In [ ]:
# Compute VaR at multiple confidence levels
var_levels = [0.01, 0.05, 0.001]  # 1%, 5%, 0.1%
var_labels = ['1%', '5%', '0.1%']

var_stable = [levy_stable.ppf(q, alpha_hat, beta_hat,
              loc=loc_hat, scale=scale_hat) for q in var_levels]
var_normal = [stats.norm.ppf(q, mu_hat, sigma_hat) for q in var_levels]
var_empirical = [np.percentile(log_ret, q * 100) for q in var_levels]

print("   VaR Comparison (left-tail quantiles):")
print(f"   {'Level':>8s}  {'Stable':>10s}  {'Normal':>10s}  {'Empirical':>10s}")
print(f"   {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}")
for lbl, vs, vn, ve in zip(var_labels, var_stable, var_normal, var_empirical):
    print(f"   {lbl:>8s}  {vs:>10.5f}  {vn:>10.5f}  {ve:>10.5f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(log_ret, bins=200, density=True, alpha=0.4, color=MAIN_BLUE,
        edgecolor='white', linewidth=0.3, label='S&P 500 log-returns')

# VaR lines
var_colors = [CRIMSON, ORANGE, FOREST]
for i, (lbl, vs, vn, ve) in enumerate(zip(var_labels, var_stable,
                                           var_normal, var_empirical)):
    ax.axvline(vs, color=var_colors[i], linewidth=1.2, linestyle='-',
               label=f'VaR {lbl} Stable ({vs:.4f})')
    ax.axvline(vn, color=var_colors[i], linewidth=1.0, linestyle='--',
               label=f'VaR {lbl} Normal ({vn:.4f})')
    ax.axvline(ve, color=var_colors[i], linewidth=0.8, linestyle=':',
               label=f'VaR {lbl} Empirical ({ve:.4f})')

ax.set_xlim(log_ret.quantile(0.0005), log_ret.quantile(0.20))
ax.set_xlabel('Log-return')
ax.set_ylabel('Density')
ax.set_title('Ch.2: VaR Comparison — Stable vs Normal vs Empirical',
             fontweight='bold', fontsize=10)

# Annotation table
table_text = 'VaR Comparison\n'
table_text += f'{"Level":>6s} {"Stable":>9s} {"Normal":>9s} {"Empir.":>9s}\n'
for lbl, vs, vn, ve in zip(var_labels, var_stable, var_normal, var_empirical):
    table_text += f'{lbl:>6s} {vs:>9.4f} {vn:>9.4f} {ve:>9.4f}\n'
ax.text(0.97, 0.97, table_text.strip(), transform=ax.transAxes,
        ha='right', va='top', fontsize=6.5, family='monospace',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3,
          frameon=False, fontsize=6)

plt.tight_layout()
save_fig('ch2_var_comparison')

## Tail Survival Function

In [ ]:
# Survival function P(X > x) on log-log scale
# Compare Stable(alpha=1.7) vs Normal
alpha_tail = 1.7
x_surv = np.logspace(-1, 2, 500)

# Stable survival function: P(X > x) = 1 - CDF(x)
sf_stable = levy_stable.sf(x_surv, alpha_tail, 0, loc=0, scale=1)
sf_normal = stats.norm.sf(x_surv, loc=0, scale=1)

# Power-law reference: C * x^{-alpha}
# For large x, stable tails ~ x^{-alpha}
C_ref = sf_stable[50] * x_surv[50]**alpha_tail  # calibrate constant
sf_power = C_ref * x_surv**(-alpha_tail)

fig, ax = plt.subplots(figsize=(8, 4))

ax.loglog(x_surv, sf_stable, color=CRIMSON, linewidth=1.3,
          label=f'Stable ($\\alpha$={alpha_tail})')
ax.loglog(x_surv, sf_normal, color=MAIN_BLUE, linewidth=1.3, linestyle='--',
          label='Normal (Gaussian)')
ax.loglog(x_surv, sf_power, color='gray', linewidth=0.8, linestyle=':',
          label=f'Power law $x^{{-{alpha_tail}}}$')

# Mark sigma thresholds
for sigma_val, lbl in [(3, '$3\\sigma$'), (5, '$5\\sigma$')]:
    ax.axvline(sigma_val, color=AMBER, linewidth=0.7, linestyle='--', alpha=0.7)
    ax.text(sigma_val * 1.05, ax.get_ylim()[1] * 0.3, lbl,
            fontsize=7, color=AMBER, va='top')

ax.set_xlabel('$x$')
ax.set_ylabel('$P(X > x)$')
ax.set_title('Ch.2: Tail Survival Function — Stable vs Normal (log-log)',
             fontweight='bold', fontsize=10)
ax.set_xlim(0.1, 100)
ax.set_ylim(1e-30, 1)

# Annotate the gap
ax.annotate('Stable tails decay\nas power law $\\sim x^{-\\alpha}$',
            xy=(10, sf_stable[np.argmin(np.abs(x_surv - 10))]),
            xytext=(20, 1e-4), fontsize=7, color=CRIMSON,
            arrowprops=dict(arrowstyle='->', color=CRIMSON, lw=0.7))
ax.annotate('Normal tails decay\nexponentially fast',
            xy=(5, sf_normal[np.argmin(np.abs(x_surv - 5))]),
            xytext=(15, 1e-15), fontsize=7, color=MAIN_BLUE,
            arrowprops=dict(arrowstyle='->', color=MAIN_BLUE, lw=0.7))

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3,
          frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_tail_survival')

## Cross-Asset Alpha Comparison

In [ ]:
# Download multiple assets and fit stable alpha
tickers = {
    'S&P 500': '^GSPC',
    'Bitcoin': 'BTC-USD',
    'EUR/USD': 'EURUSD=X',
    'Gold': 'GC=F',
    'Crude Oil': 'CL=F',
    'US 10Y': '^TNX'
}

results = []
for name, ticker in tickers.items():
    try:
        d = yf.download(ticker, start='2015-01-01', end='2025-01-01', progress=False)
        c = d['Close'].squeeze().dropna()
        r = np.log(c / c.shift(1)).dropna()
        r = r[np.isfinite(r)]  # remove inf values
        if len(r) < 100:
            print(f"   {name}: insufficient data ({len(r)} obs), skipping")
            continue
        # Fit stable distribution
        p0 = levy_stable._fitstart(r)
        params = levy_stable.fit(r, floc=p0[2], fscale=p0[3])
        alpha_est = params[0]
        beta_est = params[1]
        results.append({'Asset': name, 'Alpha': alpha_est,
                        'Beta': beta_est, 'N': len(r)})
        print(f"   {name:>12s}: alpha={alpha_est:.4f}, beta={beta_est:.4f}, n={len(r)}")
    except Exception as e:
        print(f"   {name}: error - {e}")

df_results = pd.DataFrame(results)
print(f"\n   Fitted {len(df_results)} assets successfully.")

In [ ]:
# Bar chart of alpha values
fig, ax = plt.subplots(figsize=(8, 4))

assets = df_results['Asset']
alphas_fit = df_results['Alpha']
x_pos = np.arange(len(assets))

# Color bars by alpha value (lower alpha = heavier tails = more red)
bar_colors = [CRIMSON if a < 1.7 else AMBER if a < 1.9 else FOREST
              for a in alphas_fit]

bars = ax.bar(x_pos, alphas_fit, color=bar_colors, edgecolor='white',
              linewidth=0.5, width=0.6, alpha=0.85)

# Reference line at alpha=2 (Gaussian)
ax.axhline(y=2.0, color=MAIN_BLUE, linewidth=0.8, linestyle='--',
           label='$\\alpha=2$ (Gaussian)')

# Value labels on bars
for i, (bar, val) in enumerate(zip(bars, alphas_fit)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels(assets, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Stability index $\\alpha$')
ax.set_title('Ch.2: Cross-Asset Stability Index ($\\alpha$)',
             fontweight='bold', fontsize=10)
ax.set_ylim(0, 2.3)

# Legend for color coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=CRIMSON, alpha=0.85, label='$\\alpha < 1.7$ (heavy tails)'),
    Patch(facecolor=AMBER, alpha=0.85, label='$1.7 \\leq \\alpha < 1.9$'),
    Patch(facecolor=FOREST, alpha=0.85, label='$\\alpha \\geq 1.9$ (near-Gaussian)'),
]
ax.legend(handles=legend_elements, loc='upper center',
          bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False, fontsize=7)

plt.tight_layout()
save_fig('ch2_cross_asset_alpha')